<a href="https://colab.research.google.com/github/Siddharth5723/car_policy_claims_research/blob/main/Car_damage_detection_research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q "ultralytics>=8.3.0" lightgbm opencv-python pillow pandas scikit-learn

Common Script


In [7]:
import os
import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError

Image.MAX_IMAGE_PIXELS = 100_000_000
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def collect_images(inputs):
    paths = []
    for item in inputs:
        if os.path.isdir(item):
            for name in sorted(os.listdir(item)):
                full = os.path.join(item, name)
                if os.path.isfile(full) and os.path.splitext(name)[1].lower() in VALID_EXT:
                    paths.append(full)
        elif os.path.isfile(item) and os.path.splitext(item)[1].lower() in VALID_EXT:
            paths.append(item)
    seen = set()
    unique = []
    for p in paths:
        key = os.path.realpath(p)
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def load_image(path, max_side=1280, min_side=64):
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im)
            im = im.convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError, ValueError):
        return None
    w, h = im.size
    if min(w, h) < min_side:
        return None
    scale = max_side / max(w, h)
    if scale < 1:
        im = im.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))), Image.LANCZOS)
    return np.ascontiguousarray(np.asarray(im)[:, :, ::-1])

Pipeline

In [8]:
import cv2
import numpy as np

VEHICLE_CLASS_IDS = [2, 5, 7]


def pick_device():
    try:
        import torch
        return 0 if torch.cuda.is_available() else "cpu"
    except ImportError:
        return "cpu"


def detect_vehicle(model, img, device, conf=0.35, min_area_ratio=0.15):
    res = model.predict(img, classes=VEHICLE_CLASS_IDS, conf=conf, device=device, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        return None
    h, w = img.shape[:2]
    boxes = res.boxes.xyxy.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    i = int(np.argmax(areas))
    x1, y1, x2, y2 = boxes[i]
    x1 = int(max(0, np.floor(x1)))
    y1 = int(max(0, np.floor(y1)))
    x2 = int(min(w, np.ceil(x2)))
    y2 = int(min(h, np.ceil(y2)))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    if ((x2 - x1) * (y2 - y1)) / float(w * h) < min_area_ratio:
        return None
    return img[y1:y2, x1:x2].copy(), float(confs[i]), (x1, y1, x2, y2)


def segment(model, crop, device, conf, imgsz):
    h, w = crop.shape[:2]
    res = model.predict(crop, conf=conf, imgsz=imgsz, device=device, retina_masks=True, verbose=False)[0]
    out = []
    if res.masks is None or res.boxes is None or len(res.boxes) == 0:
        return out
    masks = res.masks.data.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy().astype(int)
    confs = res.boxes.conf.cpu().numpy()
    n = min(len(masks), len(classes), len(confs))
    for k in range(n):
        m = masks[k]
        if m.shape != (h, w):
            m = cv2.resize(m.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
        out.append((int(classes[k]), float(confs[k]), m > 0.5))
    return out


def analyse_image(img, vehicle_model, parts_model, damage_model, device, cfg):
    found = detect_vehicle(vehicle_model, img, device, cfg["vehicle_conf"], cfg["min_vehicle_area"])
    if found is None:
        return None
    crop, vconf, box = found
    h, w = crop.shape[:2]
    parts = segment(parts_model, crop, device, cfg["parts_conf"], cfg["imgsz"])
    damages = segment(damage_model, crop, device, cfg["damage_conf"], cfg["imgsz"])

    damage_union = np.zeros((h, w), dtype=bool)
    for _, _, m in damages:
        damage_union |= m

    part_masks = {}
    part_confs = {}
    for cid, c, m in parts:
        name = parts_model.names.get(cid, str(cid))
        if name in part_masks:
            part_masks[name] |= m
            part_confs[name] = max(part_confs[name], c)
        else:
            part_masks[name] = m.copy()
            part_confs[name] = c

    features = {}
    for name, pm in part_masks.items():
        area = int(pm.sum())
        if area < cfg["min_part_pixels"]:
            continue
        ratio = float((pm & damage_union).sum()) / float(area)
        dconf = 0.0
        for _, c, dm in damages:
            if (dm & pm).any():
                dconf = max(dconf, c)
        features[name] = {"ratio": ratio, "damage_conf": dconf, "part_conf": part_confs[name]}

    return {
        "vehicle_conf": vconf,
        "vehicle_box": box,
        "crop": crop,
        "parts": parts,
        "damages": damages,
        "damage_union": damage_union,
        "features": features,
    }


def aggregate(per_image, part_names):
    agg = {}
    for name in part_names:
        best = None
        for r in per_image:
            f = r["features"].get(name)
            if f is None:
                continue
            if best is None or f["ratio"] > best["ratio"]:
                best = f
        if best is None:
            agg[name] = {"seen": 0, "ratio": float("nan"), "damage_conf": float("nan")}
        else:
            agg[name] = {"seen": 1, "ratio": best["ratio"], "damage_conf": best["damage_conf"]}
    return agg


def feature_vector(agg, part_names, structured=None):
    row = {}
    for name in part_names:
        row[f"{name}_ratio"] = agg[name]["ratio"]
        row[f"{name}_conf"] = agg[name]["damage_conf"]
    structured = structured or {}
    for k, v in structured.items():
        row[k] = np.nan if v is None else v
    return row


def draw_overlay(result, parts_model):
    crop = result["crop"].copy()
    tint = np.zeros_like(crop)
    tint[result["damage_union"]] = (0, 0, 255)
    out = cv2.addWeighted(crop, 1.0, tint, 0.45, 0)
    for cid, c, m in result["parts"]:
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, (0, 255, 0), 1)
        if contours:
            biggest = max(contours, key=cv2.contourArea)
            x, y, bw, bh = cv2.boundingRect(biggest)
            label = parts_model.names.get(cid, str(cid))
            cv2.putText(out, label, (x, max(10, y + 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1, cv2.LINE_AA)
    return out

RUN DEMO

In [15]:
import argparse
import json
import os
import sys

import cv2
from ultralytics import YOLO


def parse_args(args=None):
    p = argparse.ArgumentParser()
    p.add_argument("--images", nargs="+", required=True)
    p.add_argument("--parts-weights", required=True)
    p.add_argument("--damage-weights", required=True)
    p.add_argument("--vehicle-weights", default="yolov8x.pt")
    p.add_argument("--out", default="outputs")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--vehicle-conf", type=float, default=0.35)
    p.add_argument("--parts-conf", type=float, default=0.3)
    p.add_argument("--damage-conf", type=float, default=0.25)
    p.add_argument("--min-vehicle-area", type=float, default=0.15)
    p.add_argument("--min-part-pixels", type=int, default=400)
    p.add_argument("--towed", type=int, choices=[0, 1], default=None)
    p.add_argument("--drivable", type=int, choices=[0, 1], default=None)
    p.add_argument("--vehicle-age", type=float, default=None)
    return p.parse_args(args)


def run_pipeline(images, parts_weights, damage_weights, vehicle_weights="yolov8x.pt", out="outputs", imgsz=640, vehicle_conf=0.35, parts_conf=0.3, damage_conf=0.25, min_vehicle_area=0.15, min_part_pixels=400, towed=None, drivable=None, vehicle_age=None):
    for w in (parts_weights, damage_weights):
        if not os.path.isfile(w):
            print(f"missing weights file: {w}")
            return

    paths = collect_images(images)
    if not paths:
        print("no valid image files found")
        return

    os.makedirs(out, exist_ok=True)
    device = pick_device()
    vehicle_model = YOLO(vehicle_weights)
    parts_model = YOLO(parts_weights)
    damage_model = YOLO(damage_weights)
    part_names = [parts_model.names[i] for i in sorted(parts_model.names)]

    cfg = {
        "imgsz": imgsz,
        "vehicle_conf": vehicle_conf,
        "parts_conf": parts_conf,
        "damage_conf": damage_conf,
        "min_vehicle_area": min_vehicle_area,
        "min_part_pixels": min_part_pixels,
    }

    results = []
    skipped = []
    for idx, path in enumerate(paths):
        img = load_image(path)
        if img is None:
            skipped.append({"image": path, "reason": "unreadable_or_too_small"})
            continue
        try:
            r = analyse_image(img, vehicle_model, parts_model, damage_model, device, cfg)
        except Exception as e:
            skipped.append({"image": path, "reason": f"inference_error: {type(e).__name__}"})
            continue
        if r is None:
            skipped.append({"image": path, "reason": "no_vehicle_found"})
            continue
        results.append(r)
        overlay = draw_overlay(r, parts_model)
        cv2.imwrite(os.path.join(out, f"overlay_{idx:03d}.jpg"), overlay)

    if not results:
        print(json.dumps({"status": "no_usable_images", "skipped": skipped}, indent=2))
        return

    agg = aggregate(results, part_names)
    structured = {"towed": towed, "drivable": drivable, "vehicle_age": vehicle_age}
    row = feature_vector(agg, part_names, structured)

    damaged = sorted(
        [(n, v["ratio"]) for n, v in agg.items() if v["seen"] and v["ratio"] > 0.0],
        key=lambda t: t[1],
        reverse=True,
    )
    report = {
        "status": "ok",
        "images_used": len(results),
        "images_skipped": skipped,
        "damaged_parts": [{"part": n, "damaged_fraction": round(r, 4)} for n, r in damaged],
        "parts_never_seen": [n for n, v in agg.items() if not v["seen"]],
        "feature_vector": row,
    }
    with open(os.path.join(out, "report.json"), "w") as f:
        json.dump(report, f, indent=2, default=lambda o: None if o != o else float(o))
    print(json.dumps(report, indent=2, default=lambda o: None if o != o else float(o)))


def main():
    # Inside notebook environment, we prevent reading system argv directly
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-f"):
        a = parse_args()
        run_pipeline(
            images=a.images,
            parts_weights=a.parts_weights,
            damage_weights=a.damage_weights,
            vehicle_weights=a.vehicle_weights,
            out=a.out,
            imgsz=a.imgsz,
            vehicle_conf=a.vehicle_conf,
            parts_conf=a.parts_conf,
            damage_conf=a.damage_conf,
            min_vehicle_area=a.min_vehicle_area,
            min_part_pixels=a.min_part_pixels,
            towed=a.towed,
            drivable=a.drivable,
            vehicle_age=a.vehicle_age
        )
    else:
        print("Running inside a notebook environment. Use run_pipeline(...) directly to pass arguments.")


if __name__ == "__main__":
    main()

Running inside a notebook environment. Use run_pipeline(...) directly to pass arguments.


TRAIN SEGMENT

In [17]:
import argparse
import os
import sys

from ultralytics import YOLO

# 'pick_device' is already loaded in the notebook environment from the previous cells.


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--base", default="yolov8s-seg.pt")
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=-1)
    p.add_argument("--name", required=True)
    p.add_argument("--patience", type=int, default=15)

    # Inside notebook environment, we prevent reading system argv directly
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-f"):
        a = p.parse_args()
    else:
        print("Running inside a notebook environment. Please define and run training manually using the APIs.")
        return

    if not os.path.isfile(a.data):
        print(f"data yaml not found: {a.data}")
        sys.exit(1)

    device = pick_device()
    batch = a.batch
    if device == "cpu" and batch == -1:
        batch = 4

    model = YOLO(a.base)
    model.train(
        data=a.data,
        epochs=a.epochs,
        imgsz=a.imgsz,
        batch=batch,
        device=device,
        name=a.name,
        patience=a.patience,
        workers=2,
        exist_ok=True,
    )
    metrics = model.val()
    print("box mAP50-95:", metrics.box.map)
    print("mask mAP50-95:", metrics.seg.map)


if __name__ == "__main__":
    main()

Running inside a notebook environment. Please define and run training manually using the APIs.


TRAIN SEVERITY

In [20]:
import argparse
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split


def synthetic_frame(n, n_parts, seed):
    rng = np.random.default_rng(seed)
    seen = rng.random((n, n_parts)) > 0.25
    ratio = np.where(seen, rng.beta(0.3, 2.0, size=(n, n_parts)), np.nan)
    conf = np.where(seen, rng.uniform(0.3, 1.0, size=(n, n_parts)), np.nan)
    towed = rng.integers(0, 2, n)
    drivable = rng.integers(0, 2, n)
    age = rng.integers(0, 20, n).astype(float)
    score = np.nan_to_num(ratio).sum(axis=1) * 1.5 + towed + (1 - drivable) + rng.normal(0, 0.7, n)
    label = (score > np.quantile(score, 0.7)).astype(int)
    data = {}
    for i in range(n_parts):
        data[f"part{i}_ratio"] = ratio[:, i]
        data[f"part{i}_conf"] = conf[:, i]
    data["towed"] = towed
    data["drivable"] = drivable
    data["vehicle_age"] = age
    data["label"] = label
    return pd.DataFrame(data)


def fit_eval(df, cols, seed):
    X = df[cols]
    y = df["label"]
    if y.nunique() < 2:
        return None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=10, verbose=-1, random_state=seed)
    model.fit(Xtr, ytr)
    prob = model.predict_proba(Xte)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
        "auc": roc_auc_score(yte, prob),
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--csv", default=None)
    p.add_argument("--synthetic-n", type=int, default=3000)
    p.add_argument("--synthetic-parts", type=int, default=24)
    p.add_argument("--seed", type=int, default=42)

    # Safeguard against Jupyter's kernel JSON parameter
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-f"):
        a = p.parse_args()
    else:
        a = p.parse_args(args=[])

    if a.csv:
        df = pd.read_csv(a.csv)
        if "label" not in df.columns:
            print("csv must contain a 'label' column with 0/1 severity")
            sys.exit(1)
        df = df.dropna(subset=["label"])
        df["label"] = df["label"].astype(int)
    else:
        df = synthetic_frame(a.synthetic_n, a.synthetic_parts, a.seed)
        print("using synthetic data: this only verifies the plumbing, not real-world accuracy")

    if len(df) < 50:
        print("need at least 50 rows")
        sys.exit(1)

    image_cols = [c for c in df.columns if c.endswith("_ratio") or c.endswith("_conf")]
    structured_cols = [c for c in df.columns if c not in image_cols and c != "label"]
    if not image_cols or not structured_cols:
        print("need both image feature columns (*_ratio, *_conf) and structured columns")
        sys.exit(1)

    base = fit_eval(df, structured_cols, a.seed)
    full = fit_eval(df, structured_cols + image_cols, a.seed)
    if base is None or full is None:
        print("labels contain a single class")
        sys.exit(1)

    print("structured only      :", {k: round(v, 3) for k, v in base.items()})
    print("structured + image   :", {k: round(v, 3) for k, v in full.items()})


if __name__ == "__main__":
    main()

using synthetic data: this only verifies the plumbing, not real-world accuracy
structured only      : {'precision': 0.464, 'recall': 0.258, 'f1': 0.331, 'auc': np.float64(0.644)}
structured + image   : {'precision': 0.767, 'recall': 0.511, 'f1': 0.613, 'auc': np.float64(0.874)}
